<a href="https://colab.research.google.com/github/ryanaxiom/Applied-ML/blob/main/code/Day08_carnegie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student Instructions**: Before you begin, click **File > Save a copy in Drive** so you do not lose your work!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

url = ("https://raw.githubusercontent.com/ryanaxiom/Applied-ML/main/data/carnegie_data.csv")

carnegie_raw = pd.read_csv(url, encoding="cp1252")
carnegie = carnegie_raw.copy()
carnegie["research"] = carnegie["serd"] + carnegie["nonserd"]
d = carnegie[carnegie["research"].notna()].copy()
d["med_yn"] = (d["medical"] == 1).astype(int)
y = d["research"]

pile = ["fallenr20", "totdeg", "facnum", "stem_rsd", "med_yn"]
model5 = LinearRegression()
model5.fit(d[pile], y)

LinearRegression()

In [ ]:
m5pred = model5.predict(d[pile])
np.sqrt(np.mean((y - m5pred)**2))

np.float64(194050.89796792323)

In [ ]:
d["fallenr20"].corr(d["totdeg"])

np.float64(0.9642960538985523)

In [ ]:
rng = np.random.default_rng(309)
idx = rng.permutation(len(d))
A, B = d.iloc[idx[:151]], d.iloc[idx[151:]]
mA = LinearRegression()
mA.fit(A[pile], A["research"])
mB = LinearRegression()
mB.fit(B[pile], B["research"])
np.corrcoef(mA.predict(d[pile]), mB.predict(d[pile]))[0, 1]   # not sqrt(r2_score(...))

np.float64(0.987560759386111)

In [ ]:
d["locale"].value_counts().sort_index()

,count
locale,
11,114
12,59
13,45
21,41
22,7
23,7
25,1
31,4
32,12


In [ ]:
setting_map = {11:"city", 12: "city", 13: "city",
               21: "suburb", 22: "suburb", 23: "suburb",
               31: "town", 32: "town", 33: "town",
               41: "rural", 42: "rural", 43: "rural"}
d["setting"] = d["locale"].map(setting_map)
d["setting"].value_counts(dropna=False)

,count
setting,
city,218
suburb,55
town,27
NaN,1
rural,1


In [ ]:
d[d["setting"].isna()]["name"]

,name
3315,Uniformed Services University of the Health Sc...


In [ ]:
d[d["setting"]=="town"]["name"]

,name
354,Bowling Green State University-Main Campus
559,Central Michigan University
670,Clarkson University
898,Dartmouth College
1253,Georgia Southern University
1483,Indiana University of Pennsylvania-Main Campus
1802,Louisiana Tech University
1947,Miami University-Oxford
1951,Michigan Technological University
2012,Mississippi State University


Watch out!!!

In [ ]:
wrong_setting = d["locale"]//10 #floor after dividing by 10
wrong_setting.value_counts(dropna=False)

,count
locale,
1,218
2,56
3,27
4,1


In [ ]:
dd = d[d["setting"].notna()].copy()
dd = dd[dd["setting"] != "rural"]
dums = pd.get_dummies(dd["setting"],drop_first=True,dtype=int)
dums.columns.tolist()

['suburb', 'town']

In [ ]:
dd["setting"].value_counts()

,count
setting,
city,218
suburb,55
town,27


In [ ]:
dd

,unitid,name,city,stabbr,basic2000,basic2005,basic2010,basic2015,basic2018,basic2021,...,actcmp25,satacteq25,actfinal,appsf20,admitsf20,pctadmitf20,selindex,research,med_yn,setting
26,200697,Air Force Institute of Technology-Graduate Sch...,Wright-Patterson AFB,OH,59,-2,-2,17,16,16,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,35584.0,0,suburb
41,385415,Albert Einstein College of Medicine,Bronx,NY,-2,-2,-2,-2,-2,27,...,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,288654.0,1,city
102,131159,American University,Washington,DC,15,17,17,16,16,16,...,27.0,25.0,25.763524,20036.0,7744.0,0.386504,3.0,71839.0,0,city
145,104151,Arizona State University Campus Immersion,Tempe,AZ,15,15,15,15,15,15,...,21.0,22.0,21.474579,53516.0,47290.0,0.883661,2.0,585820.0,0,city
146,483124,Arizona State University Digital Immersion,Scottsdale,AZ,-2,-2,-2,17,16,16,...,0.0,0.0,0.000000,5559.0,4035.0,0.725850,0.0,87537.0,0,city
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3825,156125,Wichita State University,Wichita,KS,16,16,16,16,16,16,...,20.0,20.0,20.000000,7655.0,6096.0,0.796342,2.0,153816.0,0,city
3834,231624,William & Mary,Williamsburg,VA,16,16,16,16,16,16,...,30.0,27.0,27.888489,14201.0,5987.0,0.421590,3.0,66763.0,0,suburb
3870,168421,Worcester Polytechnic Institute,Worcester,MA,16,17,17,16,16,16,...,0.0,0.0,0.000000,11269.0,6654.0,0.590469,2.0,43092.0,0,city
3877,206604,Wright State University-Main Campus,Dayton,OH,16,16,16,17,16,16,...,18.0,18.0,18.000000,5226.0,4998.0,0.956372,1.0,49825.0,1,suburb


In [ ]:
X = pd.concat([dd[["fallenr20","med_yn"]],dums], axis=1)
model_s = LinearRegression()
model_s.fit(X,dd["research"])
dict(zip(X.columns, model_s.coef_.round(1)))

{'fallenr20': np.float64(8.0),
 'med_yn': np.float64(293789.1),
 'suburb': np.float64(-75555.0),
 'town': np.float64(-129943.0)}

In [ ]:

dums_all = pd.get_dummies(dd["setting"],dtype=int)
X_all = pd.concat([dd[["fallenr20", "med_yn"]], dums_all],axis=1)
trap = LinearRegression()
trap.fit(X_all,dd["research"])
dict(zip(X_all.columns, trap.coef_.round(0))) , trap.intercept_.round(0)

({'fallenr20': np.float64(8.0),
  'med_yn': np.float64(293789.0),
  'city': np.float64(68499.0),
  'suburb': np.float64(-7056.0),
  'town': np.float64(-61444.0)},
 np.float64(-75782.0))

In [ ]:
dumb = pd.get_dummies(d["basic2021"], prefix="basic", drop_first=True, dtype=int)
Xb = pd.concat([d[["fallenr20", "med_yn"]], dumb], axis=1)
model_b = LinearRegression()
model_b.fit(Xb, y)
r2_score(y, model_b.predict(Xb))

0.41075949427045244

In [ ]:
dict(zip(Xb.columns,model_b.coef_.round(0))), model_b.intercept_.round(0)

({'fallenr20': np.float64(6.0),
  'med_yn': np.float64(202329.0),
  'basic_16': np.float64(-263114.0),
  'basic_27': np.float64(29229.0)},
 np.float64(165483.0))

In [ ]:
d["enroll_x_med"] = d["fallenr20"] * d["med_yn"]
Xi = d[["fallenr20", "med_yn", "enroll_x_med"]]
model_i = LinearRegression()
model_i.fit(Xi, y)
model_i.coef_.round(3), model_i.intercept_.round(1)

(array([4.80500000e+00, 1.93415321e+05, 5.46700000e+00]), np.float64(19434.1))